# Lab 1: Understanding the Olist E-Commerce Dataset
## Setting Up the Machine Learning Development Environment

**Prepared by:** Sharad Laad  
**LinkedIn:** https://www.linkedin.com/in/sharadlaad/  
**Course:** Machine Learning — Unit 1: Mathematical Foundations, Data Engineering & Feature Engineering  
**Duration:** 2 Hours  
**Dataset:** Olist Brazilian E-Commerce Public Dataset

## Task 3: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid')
print('Libraries loaded successfully.')

## Task 4: Define Dataset Path

In [ ]:
# The notebook is inside notebooks/ so we go one level up with ../
DATA_DIR = Path('../data/raw')
print('DATA_DIR exists:', DATA_DIR.exists())

In [ ]:
# List all CSV files available in the raw data folder
for file in sorted(DATA_DIR.glob('*.csv')):
    size_kb = file.stat().st_size / 1024
    print(f'{file.name:55s}  {size_kb:8.1f} KB')

## Task 5: Load All Dataset Files

In [ ]:
customers         = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
geolocation       = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')
order_items       = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
payments          = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
reviews           = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
orders            = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
products          = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
sellers           = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

print('All datasets loaded!')

In [ ]:
tables = {
    'customers':          customers,
    'geolocation':        geolocation,
    'order_items':        order_items,
    'payments':           payments,
    'reviews':            reviews,
    'orders':             orders,
    'products':           products,
    'sellers':            sellers,
    'category_translation': category_translation,
}

## Task 6: Display Shape of Each Table

In [ ]:
for name, df in tables.items():
    print(f'{name:25s}: {df.shape[0]:>8,} rows  |  {df.shape[1]:>2} columns')

**Observation:**
- `customers` and `orders` each have one row per entity.
- `order_items` and `payments` can have multiple rows per order.
- `geolocation` is the largest table, containing geographic coordinates per zip code.

## Task 7: Preview Each Table

In [ ]:
customers.head()

In [ ]:
orders.head()

In [ ]:
order_items.head()

In [ ]:
products.head()

In [ ]:
sellers.head()

In [ ]:
payments.head()

In [ ]:
reviews.head()

In [ ]:
geolocation.head()

In [ ]:
category_translation.head(20)

## Task 8: Inspect Data Types

In [ ]:
customers.info()

In [ ]:
for name, df in tables.items():
    print('\n' + '=' * 80)
    print(name.upper())
    print('=' * 80)
    df.info(verbose=True, show_counts=True)

### Student Observation

| Question | Answer |
| --- | --- |
| Which columns are numeric? | `price`, `freight_value`, `payment_value`, `review_score`, `product_weight_g` |
| Which columns are categorical? | `order_status`, `payment_type`, `customer_state`, `product_category_name` |
| Which columns appear to be dates? | `order_purchase_timestamp`, `order_approved_at`, `order_delivered_*_date` |
| Which columns appear to be identifiers? | Any column ending with `_id` (e.g., `order_id`, `customer_id`, `product_id`) |

## Task 9: Create Dataset Summary Table

In [ ]:
def summarize_table(name, df):
    return {
        'table_name':    name,
        'rows':          df.shape[0],
        'columns':       df.shape[1],
        'missing_values': df.isnull().sum().sum(),
        'duplicate_rows': df.duplicated().sum(),
    }

summary = pd.DataFrame([
    summarize_table(name, df)
    for name, df in tables.items()
])

summary

In [ ]:
summary.to_csv('../reports/lab01_dataset_summary.csv', index=False)
print('Summary saved to reports/lab01_dataset_summary.csv')

## Task 10: Missing Value Analysis

In [ ]:
def missing_value_summary(df):
    missing_count   = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    result = pd.DataFrame({
        'missing_count':   missing_count,
        'missing_percent': missing_percent,
    })
    return result[result['missing_count'] > 0].sort_values(
        by='missing_percent', ascending=False
    )

missing_value_summary(orders)

In [ ]:
for name, df in tables.items():
    mv = missing_value_summary(df)
    print('\n' + '=' * 80)
    print(f'Missing Values in: {name.upper()}')
    print('=' * 80)
    if mv.empty:
        print('  No missing values found.')
    else:
        display(mv)

## Task 11: Duplicate Record Analysis

In [ ]:
for name, df in tables.items():
    duplicate_count = df.duplicated().sum()
    status = 'CLEAN' if duplicate_count == 0 else 'HAS DUPLICATES'
    print(f'{name:25s}: {duplicate_count:>6} duplicate rows  [{status}]')

**Discussion:**
Duplicate rows may indicate data entry errors, repeated system exports, or valid repeated events.
Students should not blindly delete duplicates without understanding their business meaning.

## Task 12: Explore Orders Table

In [ ]:
orders.head()

In [ ]:
orders['order_status'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
orders['order_status'].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Order Status Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Order Status')
ax.set_ylabel('Number of Orders')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../figures/lab01_order_status_distribution.png', dpi=150)
plt.show()
print('Figure saved.')

## Task 13: Explore Reviews Table

In [ ]:
reviews.head()

In [ ]:
reviews['review_score'].value_counts().sort_index()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
reviews['review_score'].value_counts().sort_index().plot(
    kind='bar', ax=ax, color='mediumseagreen', edgecolor='white'
)
ax.set_title('Review Score Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Review Score (1 = Worst, 5 = Best)')
ax.set_ylabel('Number of Reviews')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../figures/lab01_review_score_distribution.png', dpi=150)
plt.show()

## Task 14: Explore Payments Table

In [ ]:
payments.head()

In [ ]:
payments['payment_type'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
payments['payment_type'].value_counts().plot(
    kind='bar', ax=ax, color='coral', edgecolor='white'
)
ax.set_title('Payment Type Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Payment Type')
ax.set_ylabel('Number of Payments')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('../figures/lab01_payment_type_distribution.png', dpi=150)
plt.show()

## Task 15: Explore Products Table

In [ ]:
products.head()

In [ ]:
missing_value_summary(products)

In [ ]:
print('Unique product categories:', products['product_category_name'].nunique())

In [ ]:
products['product_category_name'].value_counts().head(10)

## Task 16: Explore Customers Table

In [ ]:
customers.head()

In [ ]:
customers['customer_state'].value_counts().head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
customers['customer_state'].value_counts().head(10).plot(
    kind='bar', ax=ax, color='slateblue', edgecolor='white'
)
ax.set_title('Top 10 Customer States', fontsize=14, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Number of Customers')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../figures/lab01_customer_states.png', dpi=150)
plt.show()

## Task 17: Initial Business Questions

### Q1: How many unique customers are present?

In [ ]:
n_unique = customers['customer_unique_id'].nunique()
print(f'Unique customers: {n_unique:,}')

### Q2: How many orders are present?

In [ ]:
n_orders = orders['order_id'].nunique()
print(f'Total orders: {n_orders:,}')

### Q3: Most common order statuses?

In [ ]:
orders['order_status'].value_counts()

### Q4: Most common payment methods?

In [ ]:
payments['payment_type'].value_counts()

### Q5: Average review score?

In [ ]:
avg_score = reviews['review_score'].mean()
print(f'Average review score: {avg_score:.2f} / 5.0')

### Q6: Which states have the most customers?

In [ ]:
customers['customer_state'].value_counts().head(10)

## Task 18: Identify Potential Machine Learning Problems

| ML Problem | Type | Target |
| --- | --- | --- |
| Predict whether an order will be delivered late | Classification | `is_late` (1 = late, 0 = on time) |
| Predict customer review score | Regression / Classification | `review_score` |
| Predict payment value | Regression | `payment_value` |
| Segment customers by purchase behaviour | Clustering | No explicit target |
| Recommend product categories | Recommendation | Product category |
| Detect unusual orders (fraud) | Anomaly Detection | No explicit target |
| Forecast monthly sales revenue | Time Series | Monthly revenue |

## Task 19: Initial Data Dictionary

The full data dictionary is stored in `reports/lab01_data_dictionary.md`.
A summary of the three most important tables is shown below.

### Table 1: customers

| Column | Data Type | Description | Example |
| --- | --- | --- | --- |
| customer_id | String | Unique identifier per order (not per person) | 06b8999e... |
| customer_unique_id | String | Unique identifier for the actual person | 861eff47... |
| customer_zip_code_prefix | Integer | Zip code prefix | 14409 |
| customer_city | String | City of the customer | franca |
| customer_state | String | State code | SP |

### Table 2: orders

| Column | Data Type | Description | Example |
| --- | --- | --- | --- |
| order_id | String | Unique order identifier | e481f51c... |
| customer_id | String | Links to customers table | 9ef432eb... |
| order_status | String | Current order status | delivered |
| order_purchase_timestamp | Timestamp | When the order was placed | 2017-10-02 10:56:33 |
| order_approved_at | Timestamp | Payment approval time | 2017-10-02 11:07:15 |
| order_delivered_customer_date | Timestamp | Actual delivery date | 2017-10-10 22:36:38 |
| order_estimated_delivery_date | Timestamp | Estimated delivery date | 2017-10-18 00:00:00 |

### Table 3: order_items

| Column | Data Type | Description | Example |
| --- | --- | --- | --- |
| order_id | String | Links to orders table | 00010242... |
| order_item_id | Integer | Item sequence number in this order | 1 |
| product_id | String | Links to products table | 4244733e... |
| seller_id | String | Links to sellers table | 48436dad... |
| price | Float | Item price in BRL | 58.90 |
| freight_value | Float | Shipping cost | 13.29 |

## Task 20: README File
The `README.md` file has been created in the project root.
It describes the project, dataset, folder structure, and setup instructions.

## Task 21: Git and GitHub Setup

The Git repository has been initialized. Run the following from the terminal:

```bash
git init
git add .
git commit -m 'Add Lab 1 dataset exploration notebook'
git remote add origin https://github.com/username/MachineLearning2026.git
git branch -M main
git push -u origin main
```

> **Note:** The `.gitignore` file is already configured to exclude `data/raw/`, `models/`, `data/processed/`, `.venv/`, etc.

## Extension Activity: Merging Tables

In [ ]:
# Merge orders with customers
orders_customers = orders.merge(customers, on='customer_id', how='left')
print('orders + customers shape:', orders_customers.shape)
orders_customers.head()

In [ ]:
# Merge orders with reviews
orders_reviews = orders.merge(reviews, on='order_id', how='left')
print('orders + reviews shape:', orders_reviews.shape)
orders_reviews.head()

In [ ]:
# Average review score by state
score_by_state = (
    orders_customers
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
    .groupby('customer_state')['review_score']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'avg_score', 'count': 'n_reviews'})
    .sort_values('avg_score', ascending=False)
)
score_by_state.head(10)

In [ ]:
# Order count by month
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['year_month'] = orders['order_purchase_timestamp'].dt.to_period('M')
monthly_orders = orders.groupby('year_month').size().rename('order_count')

fig, ax = plt.subplots(figsize=(14, 5))
monthly_orders.plot(ax=ax, marker='o', color='steelblue')
ax.set_title('Monthly Order Count', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../figures/lab01_monthly_orders.png', dpi=150)
plt.show()

In [ ]:
# Top 10 product categories by number of orders
top_categories = (
    order_items
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .groupby('product_category_name')['order_id']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
top_categories.plot(kind='bar', ax=ax, color='darkorange', edgecolor='white')
ax.set_title('Top 10 Product Categories by Orders', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Number of Orders')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../figures/lab01_top_categories.png', dpi=150)
plt.show()

## Reflection Questions

**1. Why is it important to understand the dataset before building a Machine Learning model?**

Understanding the dataset is essential to avoid the 'garbage in, garbage out' problem. It helps identify missing values, outliers, the distribution of features, and the relationships between tables. It ensures that the features we engineer are meaningful, and that our model is built on solid, clean, and reliable data.

**2. Which table appears to be the central table in the Olist dataset? Why?**

The `orders` table is the central table because it connects all other tables: customers via `customer_id`, order items via `order_id`, payments via `order_id`, and reviews via `order_id`. Almost all analysis begins by filtering or aggregating from `orders`.

**3. What is the difference between `customer_id` and `customer_unique_id`?**

`customer_id` is generated per order — a single customer who places 3 separate orders will appear 3 times with 3 different `customer_id` values. `customer_unique_id` is a persistent identifier for the actual person across all their orders.

**4. Which columns appear to represent dates?**

`order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`, `review_creation_date`, `review_answer_timestamp`, `shipping_limit_date`.

**5. Which columns appear to represent categorical variables?**

`order_status`, `payment_type`, `customer_state`, `customer_city`, `product_category_name`, `geolocation_state`.

**6. Which columns contain missing values?**

The `orders` table has missing values in the delivery date columns (`order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`) for non-delivered orders. The `products` table has missing values in `product_category_name` and physical dimensions. The `reviews` table has missing values in `review_comment_title` and `review_comment_message` since not all customers leave written comments.

**7. Which tables can be joined using `order_id`?**

`orders`, `order_items`, `payments`, `reviews`.

**8. Which tables can be joined using `product_id`?**

`order_items` and `products`.

**9. What are three potential Machine Learning problems?**

1. **Late delivery prediction** — Binary classification using delivery timestamps and geolocation.
2. **Review score prediction** — Regression/classification using delivery performance and product info.
3. **Customer segmentation** — Unsupervised clustering based on purchase history and order value.

**10. What ethical concerns arise when using customer data for ML?**

- **Privacy:** Geographic and purchase data can re-identify individuals, even without names.
- **Algorithmic bias:** Models may learn to offer worse service to certain states or neighborhoods.
- **Transparency:** Customers are rarely informed how their data is used for ML.
- **Fairness:** Pricing or delivery prioritization models could discriminate against certain groups.
- **Data security:** Storing and processing customer data creates security obligations.

---
## Key Takeaways

- Machine Learning begins with **data understanding**, not model training.
- Real-world datasets are **relational, messy, and incomplete**.
- Proper **project structure** improves reproducibility.
- **Exploratory Data Analysis** is essential before any model building.
- **Data quality** directly affects model quality.
- GitHub helps build a **professional ML portfolio**.

---
*End of Lab 1 — Machine Learning Engineering Studio 1*

*Prepared by Sharad Laad | ORY AI Labs*